# Creating the training/validation/test dataset USING ONLY **ROI IMAGES** AND **LABELS**

# Important libraries

# Preprocessing class with all needed functions

-------------------------------------------------------------------------------------------------------------------------------------------------------

In [1]:
import os
import cv2
import glob
import numpy as np
from tqdm import tqdm

class CBIS_ROI_ClassifierPreprocessor:
    def __init__(self, img_size=(224, 224)):
        """
        Preprocessor for CBIS-DDSM ROI classification (calc vs mass).
        """
        self.img_size = img_size

        # Class mapping (you can extend later for benign/malignant)
        self.class_mapping = {
            'calc': 0,
            'mass': 1
        }

    def load_image(self, image_path):
        """Load an image in grayscale or color."""
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Could not load image: {image_path}")
        return img

    def enhance_contrast(self, img):
        """Enhance image contrast using CLAHE (good for mammograms)."""
        if img.dtype != np.uint8:
            img = (img * 255).astype(np.uint8)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(img)
        return enhanced

    def resize_with_padding(self, img):
        """Resize with padding to keep aspect ratio."""
        h, w = img.shape[:2]
        target_h, target_w = self.img_size
        scale = min(target_w / w, target_h / h)
        new_w, new_h = int(w * scale), int(h * scale)
        resized_img = cv2.resize(img, (new_w, new_h))
        padded_img = np.zeros((target_h, target_w), dtype=resized_img.dtype)
        x_offset = (target_w - new_w) // 2
        y_offset = (target_h - new_h) // 2
        padded_img[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized_img
        return padded_img

    def process_image(self, image_path):
        """Load, enhance, and resize a single ROI image."""
        img = self.load_image(image_path)
        img = self.enhance_contrast(img)
        img = self.resize_with_padding(img)
        return img

    def load_dataset_from_folder(self, folder_path, label_name):
        """
        Load all ROI images from a folder tree (e.g., .../roi_crops/),
        assign the class label based on 'label_name' (calc/mass).
        """
        print(f"Loading {label_name} images from: {folder_path}")
        all_image_paths = glob.glob(os.path.join(folder_path, "**", "*.png"), recursive=True)
        X = []
        y = []
        for image_path in tqdm(all_image_paths, desc=f"Processing {label_name}"):
            try:
                processed = self.process_image(image_path)
                X.append(processed)
                y.append(self.class_mapping[label_name])
            except Exception as e:
                print(f"Error processing {image_path}: {e}")

        X = np.array(X)
        y = np.array(y)
        # Add channel dimension (H, W, 1)
        X = np.expand_dims(X, axis=-1)
        print(f"Loaded {len(X)} images for class '{label_name}' -> shape {X.shape}")
        return X, y


In [2]:
base_dir = r"D:\cbis-ddsm_dataset_licenta\data\processed_separate_roi_full_masks"

calc_train_dir = os.path.join(base_dir, "calc_case_description_train_set_png", "roi_crops")
calc_test_dir  = os.path.join(base_dir, "calc_case_description_test_set_png", "roi_crops")
mass_train_dir = os.path.join(base_dir, "mass_case_description_train_set_png", "roi_crops")
mass_test_dir  = os.path.join(base_dir, "mass_case_description_test_set_png", "roi_crops")

preprocessor = CBIS_ROI_ClassifierPreprocessor(img_size=(224, 224))

# === Load all subsets ===
X_train_calc, y_train_calc = preprocessor.load_dataset_from_folder(calc_train_dir, "calc")
X_test_calc,  y_test_calc  = preprocessor.load_dataset_from_folder(calc_test_dir,  "calc")
X_train_mass, y_train_mass = preprocessor.load_dataset_from_folder(mass_train_dir, "mass")
X_test_mass,  y_test_mass  = preprocessor.load_dataset_from_folder(mass_test_dir,  "mass")


Loading calc images from: D:\cbis-ddsm_dataset_licenta\data\processed_separate_roi_full_masks\calc_case_description_train_set_png\roi_crops


Processing calc: 100%|████████████████████████████████████████████████████████████| 1546/1546 [00:06<00:00, 249.45it/s]


Loaded 1546 images for class 'calc' -> shape (1546, 224, 224, 1)
Loading calc images from: D:\cbis-ddsm_dataset_licenta\data\processed_separate_roi_full_masks\calc_case_description_test_set_png\roi_crops


Processing calc: 100%|██████████████████████████████████████████████████████████████| 326/326 [00:01<00:00, 297.12it/s]


Loaded 326 images for class 'calc' -> shape (326, 224, 224, 1)
Loading mass images from: D:\cbis-ddsm_dataset_licenta\data\processed_separate_roi_full_masks\mass_case_description_train_set_png\roi_crops


Processing mass: 100%|█████████████████████████████████████████████████████████████| 1318/1318 [00:42<00:00, 30.93it/s]


Loaded 1318 images for class 'mass' -> shape (1318, 224, 224, 1)
Loading mass images from: D:\cbis-ddsm_dataset_licenta\data\processed_separate_roi_full_masks\mass_case_description_test_set_png\roi_crops


Processing mass: 100%|██████████████████████████████████████████████████████████████| 378/378 [00:00<00:00, 456.31it/s]

Loaded 378 images for class 'mass' -> shape (378, 224, 224, 1)


In [3]:
print(X_train_calc.shape)
print(X_test_calc.shape)
print(X_train_mass.shape)
print(X_test_mass.shape)

(1546, 224, 224, 1)
(326, 224, 224, 1)
(1318, 224, 224, 1)
(378, 224, 224, 1)
